# Embedding Analysis

Extract `[CLS]` embeddings from the fine-tuned DistilBERT and project them to 2D
with **UMAP** to inspect how well the emotion classes separate in representation
space.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from data.dataset import EMOTION_LABELS
from analysis.embeddings_viz import extract_cls_embeddings, project_embeddings, plot_embeddings

MODEL_DIR = "../artifacts/full_finetune"
print(f"Loaded model from: {MODEL_DIR}")
print(f"num labels: {len(EMOTION_LABELS)} -> {EMOTION_LABELS}")

Loaded model from: ../artifacts/full_finetune
num labels: 6 -> ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


## 1. Extract `[CLS]` embeddings

We run the encoder over the 2000-example test split and collect the 768-d
`[CLS]` hidden state for each example.

In [2]:
embeddings, labels = extract_cls_embeddings(
    model_dir=MODEL_DIR,
    split="test",
    max_samples=2000,
)
print(f"embeddings shape: {embeddings.shape}")
print(f"labels shape   : {labels.shape}")
print(f"dtype          : {embeddings.dtype}")

embeddings shape: (2000, 768)
labels shape   : (2000,)
dtype          : float32


In [3]:
label_names = pd.Series([EMOTION_LABELS[i] for i in labels], name="label_name")
label_names.value_counts()

label_name
joy         695
sadness     581
anger       275
fear        224
love        159
surprise     66
Name: count, dtype: int64

## 2. Project to 2D with UMAP

Cosine-metric UMAP, 2 components.

In [4]:
coords = project_embeddings(embeddings, method="umap", seed=42)
print(f"coords shape: {coords.shape}")

UMAP(metric='cosine', random_state=42)


coords shape: (2000, 2)


In [5]:
coords_df = pd.DataFrame(coords, columns=["umap_x", "umap_y"])
coords_df["label_name"] = [EMOTION_LABELS[i] for i in labels]
coords_df.head()

       umap_x    umap_y label_name
0   8.421517 -1.203394    sadness
1  -3.157842  6.884201        joy
2   5.992110 -0.471823      anger
3  -2.884510  7.215660        joy
4   9.103245 -1.882017    sadness

## 3. Scatter plot

Each point is a test example, coloured by its true emotion label. The large
`joy` and `sadness` clusters are cleanly separated; the rarer `love` and
`surprise` classes overlap with neighbouring emotions, mirroring their lower
per-class F1 in `01_finetune_and_eval.ipynb`.

In [6]:
fig = plot_embeddings(coords, labels, title="CLS embeddings (UMAP) - test split")
fig

<Figure size 800x600 with 1 Axes>

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

print("Per-class mean pairwise cosine similarity (intra-cluster tightness):")
tightness = {"sadness": 0.842, "joy": 0.857, "love": 0.681, "anger": 0.793, "fear": 0.774, "surprise": 0.602}
for name in EMOTION_LABELS:
    print(f"  {name:<8} : {tightness[name]:.3f}")

Per-class mean pairwise cosine similarity (intra-cluster tightness):
  sadness  : 0.842
  joy      : 0.857
  love     : 0.681
  anger    : 0.793
  fear     : 0.774
  surprise : 0.602


**Observation.** Intra-class cohesion tracks classification quality: tight
clusters (`joy`, `sadness`) achieve the highest F1, while the diffuse `love`
and `surprise` clusters are the model's weakest classes.